## EDA For ML

## Objective

The objective of this notebook is to perform Exploratory Data Analysis (EDA) on the ML-ready Gold dataset to:

- Understand the dataset structure.
- Analyze missing values.
- Study feature distributions.
- Identify outliers.
- Explore relationships between variables.
- Generate business insights.
- Prepare the data for feature engineering and model building.

#### Step 1: Load the Cleaned ML Dataset

In [0]:
ml_df = spark.read.table('retail_project.gold.ml_cleaned_sales_data')
display(ml_df)

In [0]:
#Checking the Rows and columns of the dataframe
print("Rows :", ml_df.count())
print("Columns :", len(ml_df.columns))

#### Step 2 — Verify Schema Again

In [0]:
ml_df.printSchema()


#### Step 3 — Checking Missing Values After cleaned Dataset

In [0]:
from pyspark.sql.functions import col, count, when

null_df = ml_df.select([
    count(when(col(column).isNull(), column)).alias(column)
    for column in ml_df.columns
])

display(null_df)

#### Step 4 — Summary Statistics

In [0]:
ml_df.describe().show()

#### **Inference from Summary Statistics**

* The cleaned dataset contains **3,717 records** with **no missing values**.
* The data covers **2025 (January–March)**.
* The average **Total Revenue** is **75,920.09**, ranging from **1,015.00** to **239,685.00**, showing high variation in sales.
* Customers purchase an average of **2 products per order**, with a maximum of **5 products**.
* Product prices range from **1,007.00** to **79,959.00**, indicating both low-priced and premium products.
* The dataset is **clean, consistent, and ready for visualization, feature engineering, and machine learning**.


#### Step 5 — Identify Numeric Columns

In [0]:
#storing the columns in a list for future used for visualization and correlation.
numeric_cols = [
    "Total_revenue",
    "Total_quantity",
    "Total_orders",
    "Avg_revenue",
    "Avg_quantity",
    "Min_price",
    "Max_price"
]

In [0]:
display(numeric_cols)

In [0]:
display(ml_df.select(numeric_cols))

#### Step 6 — Distribution Analysis

- This is one of the most important EDA steps.

We'll examine:

- Revenue
- Quantity

Since Databricks doesn't have a built-in histogram command for Spark DataFrames, convert only the required column to Pandas for plotting.

In [0]:
# Create a Pandas DataFrame from the Spark DataFrame for visualization purposes total_revenue
revenue_pd = ml_df.select("Total_revenue").toPandas()
display(revenue_pd)


In [0]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.hist(revenue_pd["Total_revenue"], bins=30)

plt.title("Revenue Distribution")
plt.xlabel("Revenue")
plt.ylabel("Frequency")
plt.show()

#### Why Convert to Pandas here?

- For **ETL, data cleaning, and feature engineering**, use **PySpark** to efficiently process large datasets.

- For **data visualization**, converting a **small dataset (a few thousand rows)** to Pandas is completely acceptable because it enables easy plotting with libraries such as Matplotlib and Seaborn.

- This is why **PySpark and Pandas are commonly used together** in Databricks notebooks—PySpark for scalable data processing and Pandas for visualization and analysis.

#### Step 7 — Category Analysis

In [0]:
# sum of revenue for each category
from pyspark.sql.functions import round, sum
display(ml_df.groupBy('category').agg(round(sum('Total_revenue'), 2).alias("Total_revenue")))


Questions:

- Which category generates the highest revenue?
- Which category contributes the least?

#### **Inference**

* **Electronics** generated the **highest total revenue (125.65M)**, making it the top-performing product category.
* **Home & Living** generated **79.64M** in revenue, ranking second.
* **Fashion** contributed **76.90M**, making it the lowest revenue-generating category among the three.
* **Electronics** is the primary revenue driver and presents the greatest opportunity for business growth and marketing initiatives.


In [0]:
# Revenue by category in visualization
from pyspark.sql.functions import sum, round
import matplotlib.pyplot as plt

# Aggregate revenue by category
category_revenue = (
    ml_df.groupBy("category")
         .agg(round(sum("Total_revenue"), 2).alias("Total_revenue"))
         .toPandas()
)

# Create bar chart
plt.figure(figsize=(8, 5))
plt.bar(category_revenue["category"], category_revenue["Total_revenue"])

plt.title("Total Revenue by Category")
plt.xlabel("Category")
plt.ylabel("Total Revenue")

# Display values on bars
for i, value in enumerate(category_revenue["Total_revenue"]):
    plt.text(i, value, f"{value/1e6:.1f}M", ha="center", va="bottom")

plt.show()

#### Step 8:  Monthly Revenue Trend

In [0]:
# Monthly trend of revenue
ml_df.groupby("month").agg(round(sum("Total_revenue"), 2).alias("Total_revenue")).display()

In [0]:
# Revenue by month in visualization
from pyspark.sql.functions import sum, round
import seaborn as sns
import matplotlib.pyplot as plt

# Aggregate revenue by month
monthly_revenue = (
    ml_df.groupBy("Month")
         .agg(round(sum("Total_revenue"), 2).alias("Total_revenue"))
         .orderBy("Month")
         .toPandas()
)

# Plot
plt.figure(figsize=(8, 5))

ax = sns.barplot(
    data=monthly_revenue,
    x="Month",
    y="Total_revenue"
)

# Add value labels (in Millions)
for container in ax.containers:
    ax.bar_label(
        container,
        labels=[f"{v/1e6:.2f}M" for v in monthly_revenue["Total_revenue"]],
        padding=3
    )

plt.title("Total Revenue by Month")
plt.xlabel("Month")
plt.ylabel("Total Revenue")
plt.xticks([0, 1, 2], ["January", "February", "March"])

plt.show()

### **Inference**

* **January** generated the **highest revenue (100.11M)**, making it the best-performing month.
* **March** recorded the **second-highest revenue (97.72M)**, showing a recovery after February.
* **February** generated the **lowest revenue (84.36M)** among the three months.
* Overall, revenue remained **strong and consistent** across all three months, with a noticeable dip in **February**.


#### Step 9: Gender Analysis

In [0]:
# Revenue by gender
ml_df.groupBy('gender').agg(round(sum('Total_revenue'), 2).alias("Total_revenue")).display()

In [0]:
# Revenue by gender in visualization
from pyspark.sql.functions import sum, round
import seaborn as sns
import matplotlib.pyplot as plt

# Aggregate revenue by gender
gender_revenue = (
    ml_df.groupBy("gender")
         .agg(round(sum("Total_revenue"), 2).alias("Total_revenue"))
         .toPandas()
)

# Plot
plt.figure(figsize=(6, 5))

ax = sns.barplot(
    data=gender_revenue,
    x="gender",
    y="Total_revenue"
)

# Add value labels (Millions)
for container in ax.containers:
    ax.bar_label(
        container,
        labels=[f"{v/1e6:.2f}M" for v in gender_revenue["Total_revenue"]],
        padding=3
    )

plt.title("Total Revenue by Gender")
plt.xlabel("Gender")
plt.ylabel("Total Revenue")
plt.xticks([0, 1, 2], ["Male", "Unknown", "Female"])

plt.show()

### **Inference**

* **Male (M)** customers generated the **highest total revenue (125.16M)**.
* **Female (F)** customers contributed **119.93M**, which is close to the revenue generated by male customers.
* **Unknown (U)** gender contributed the **lowest revenue (37.10M)**, likely due to missing or unspecified gender information.
* Overall, **male and female customers contribute almost equally to total revenue**, while the **Unknown** category accounts for a much smaller share.


#### Step 10: State Analysis

In [0]:
# Revenue by State
ml_df.groupBy('state').agg(round(sum('Total_revenue'), 2).alias('Total_revenue')).orderBy('Total_revenue', ascending=False).display()

### **Inference**

* **Unknown** state generated the **highest total revenue (37.10M)**, indicating that many records have missing or unspecified state information.
* Among the known states, **West Bengal (32.87M)** generated the highest revenue.
* **Uttar Pradesh (32.26M)** and **Karnataka (31.89M)** are the next top-performing states.
* **Telangana**, **Tamil Nadu**, and **Delhi** each contributed around **30–31M**, showing relatively consistent sales.
* **Maharashtra (26.76M)** generated the **lowest revenue** among the listed states.
* Revenue is fairly well distributed across states, with no single state dominating overall sales.


In [0]:
from pyspark.sql.functions import sum, round
import seaborn as sns
import matplotlib.pyplot as plt

# Aggregate revenue by state
state_revenue = (
    ml_df.groupBy("state")
         .agg(round(sum("Total_revenue"), 2).alias("Total_revenue"))
         .orderBy("Total_revenue", ascending=False)
         .toPandas()
)

# Plot
plt.figure(figsize=(10, 6))

ax = sns.barplot(
    data=state_revenue,
    y="state",
    x="Total_revenue"
)

# Add value labels (Millions)
for container in ax.containers:
    ax.bar_label(
        container,
        labels=[f"{v/1e6:.2f}M" for v in state_revenue["Total_revenue"]],
        padding=3
    )

plt.title("Total Revenue by State")
plt.xlabel("Total Revenue")
plt.ylabel("State")

plt.show()

Why a Horizontal Bar Chart?

- ✅ State names are longer and easier to read horizontally.
- ✅ States are sorted from highest to lowest revenue.
- ✅ This is the preferred visualization style for categorical variables with many categories in EDA and business dashboards.

#### Step 11: City Analysis


In [0]:
# Revenue by City
ml_df.groupBy('city').agg(round(sum('Total_revenue'), 2).alias('Total_revenue')).orderBy('Total_revenue', ascending=False).display()

### **Inference**

* **Unknown** city generated the **highest total revenue (37.10M)**, indicating that some customer location information is missing.
* Among the known cities, **Mumbai (31.12M)** generated the highest revenue.
* **Bangalore (30.92M)** and **Kolkata (29.85M)** are the next top-performing cities.
* **Pune**, **Delhi**, **Chennai**, **Lucknow**, and **Jaipur** contributed relatively similar revenue, ranging from **25M–28M**.
* **Hyderabad (18.95M)** generated the **lowest revenue** among the listed cities.
* Overall, revenue is well distributed across major cities, with **Mumbai and Bangalore** leading sales among known locations.



In [0]:
### Revenue by City in visualization
from pyspark.sql.functions import sum, round
import seaborn as sns
import matplotlib.pyplot as plt

# Aggregate revenue by city
city_revenue = (
    ml_df.groupBy("city")
         .agg(round(sum("Total_revenue"), 2).alias("Total_revenue"))
         .orderBy("Total_revenue", ascending=False)
         .toPandas()
)

# Plot
plt.figure(figsize=(10, 6))

ax = sns.barplot(
    data=city_revenue,
    y="city",
    x="Total_revenue"
)

# Add value labels (Millions)
for container in ax.containers:
    ax.bar_label(
        container,
        labels=[f"{v/1e6:.2f}M" for v in city_revenue["Total_revenue"]],
        padding=3
    )

plt.title("Total Revenue by City")
plt.xlabel("Total Revenue")
plt.ylabel("City")

plt.show()


#### Step 12: Correlation Analysis

- Spark doesn't provide a full correlation matrix directly, so convert only the numeric columns.

In [0]:
numeric_pd = ml_df.select(numeric_cols).toPandas()
display(numeric_pd)

In [0]:
corr = numeric_pd.corr().round(2)
display(corr)

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

# Compute correlation matrix
corr = numeric_pd.corr().round(2)

# Plot heatmap
plt.figure(figsize=(10, 8))

sns.heatmap(
    corr,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    linewidths=0.5,
    square=True
)

plt.title("Correlation Matrix")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)

plt.tight_layout()
plt.show()

### **Inference from Correlation Matrix**

* **`Total_revenue` and `Avg_revenue`** have a **perfect positive correlation (1.00)**, indicating they contain the same information. One of these features should be dropped to avoid redundancy.
* **`Total_quantity` and `Avg_quantity`** have a **very strong positive correlation (0.99)**, suggesting they are almost identical. Retaining only one of them is recommended.
* **`Min_price` and `Max_price`** have a **perfect positive correlation (1.00)**, indicating both features carry identical information. Keeping one is sufficient.
* **`Total_revenue`** has a **strong positive correlation (0.76)** with both **`Min_price`** and **`Max_price`**, showing that higher-priced products tend to generate higher revenue.
* **`Total_revenue`** has a **moderate positive correlation (0.54)** with **`Total_quantity`**, indicating that purchasing more items generally increases revenue.
* **`Total_orders`** has **very weak or no correlation** with the other features, suggesting it has little influence on revenue in this dataset.

### **Key Insight**

* Remove highly correlated features (`Avg_revenue`, `Avg_quantity`, and either `Min_price` or `Max_price`) to reduce multicollinearity and simplify the machine learning model.


In [0]:
# Dropping the highly correlated columns from the Dataframe to avoid multicollinearity
ml_df = ml_df.drop('Avg_revenue', 'Avg_quantity', 'Min_price')
ml_df.display()

#### Updating the Numeric columns


In [0]:
#Update the list of numeric columns
numeric_cols = ['Total_revenue', 'Total_quantity', 'Total_orders', 'Max_price']
display(numeric_cols)

#### Step 13: Outlier Detection

we have 2 ways of detecting outliers :

- Visualize Outliers Using Boxplots
- IQR Method

#### Visualize Outliers Using Boxplots

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

# Convert required columns to Pandas
numeric_pd = ml_df.select(numeric_cols).toPandas()

# Create boxplots
plt.figure(figsize=(12, 6))

for i, col in enumerate(numeric_cols, 1):
    plt.subplot(2, 2, i)
    sns.boxplot(y=numeric_pd[col])
    plt.title(col)

plt.tight_layout()
plt.show()

#### Detect Outliers Using the IQR Method

In [0]:
from pyspark.sql.functions import col

for column in numeric_cols:

    # Calculate Q1 and Q3
    Q1 = ml_df.approxQuantile(column, [0.25], 0.01)[0]
    Q3 = ml_df.approxQuantile(column, [0.75], 0.01)[0]

    # Calculate IQR
    IQR = Q3 - Q1

    # Calculate bounds
    lower_bound = Q1 - (1.5 * IQR)
    upper_bound = Q3 + (1.5 * IQR)

    # Count outliers
    outliers = ml_df.filter(
        (col(column) < lower_bound) |
        (col(column) > upper_bound)
    ).count()

    print(f"{column}")
    print(f"Q1           : {Q1}")
    print(f"Q3           : {Q3}")
    print(f"IQR          : {IQR}")
    print(f"Lower Bound  : {lower_bound}")
    print(f"Upper Bound  : {upper_bound}")
    print(f"Outliers     : {outliers}")
    print("-" * 50)

#### Create an Outlier Summary Table

In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import col
import builtins

outlier_summary = []

for column in numeric_cols:

    Q1 = ml_df.approxQuantile(column, [0.25], 0.01)[0]
    Q3 = ml_df.approxQuantile(column, [0.75], 0.01)[0]

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    count = ml_df.filter(\
        (col(column) < lower) |\
        (col(column) > upper)\
    ).count()

    outlier_summary.append(
        Row(
            Feature=column,
            Q1=builtins.round(Q1, 2),
            Q3=builtins.round(Q3, 2),
            IQR=builtins.round(IQR, 2),
            Lower_Bound=builtins.round(lower, 2),
            Upper_Bound=builtins.round(upper, 2),
            Outlier_Count=count
        )
    )

display(spark.createDataFrame(outlier_summary))

### **Inference from Outlier Detection (IQR Method)**

* **`Total_revenue`** contains **17 outliers**, indicating a few transactions with exceptionally high revenue. Since this is a retail sales dataset, these are likely to be **genuine high-value purchases** rather than data errors.
* **`Total_quantity`** has **no outliers**, showing that the quantity values are consistent across the dataset.
* **`Total_orders`** has **6 outliers** because most customers placed **1 order**, while a few placed **2 orders**. These represent valid business scenarios and do not require removal.
* **`Max_price`** has **no outliers**, indicating that product prices fall within the expected range.

### **Key Insights**

* ✅ Only **`Total_revenue`** shows a small number of outliers (**17 records**, less than **0.5%** of the dataset).
* ✅ The outliers are likely **valid business transactions**, not incorrect data.
* ✅ Therefore, **no outlier removal is recommended**, and the dataset is suitable for machine learning.


### **Inference from Outlier Detection (IQR Method)**

* **`Total_revenue`** contains **17 outliers**, indicating a few transactions with exceptionally high revenue. Since this is a retail sales dataset, these are likely to be **genuine high-value purchases** rather than data errors.
* **`Total_quantity`** has **no outliers**, showing that the quantity values are consistent across the dataset.
* **`Total_orders`** has **6 outliers** because most customers placed **1 order**, while a few placed **2 orders**. These represent valid business scenarios and do not require removal.
* **`Max_price`** has **no outliers**, indicating that product prices fall within the expected range.

### **Key Insights**

* ✅ Only **`Total_revenue`** shows a small number of outliers (**17 records**, less than **0.5%** of the dataset).
* ✅ The outliers are likely **valid business transactions**, not incorrect data.
* ✅ Therefore, **no outlier removal is recommended**, and the dataset is suitable for machine learning.


In [0]:
ml_df.write.format('delta').mode('overwrite').saveAsTable('retail_project.gold.ml_sales_data_afterEDA')
